<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_1/HuggingFace_Tokenizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция: HuggingFace Tokenizers — практическое обучение и использование субсловных токенизаторов

## 1. Введение

В предыдущих лекциях мы изучили теоретические основы трёх основных алгоритмов субсловной токенизации: BPE, WordPiece и Unigram Language Model. Каждый из них имеет свои особенности и области применения. Однако теория без практики остаётся неполной. В реальных проектах по обработке естественного языка токенизаторы редко реализуются с нуля; вместо этого используются готовые библиотеки, которые предоставляют эффективные, протестированные и оптимизированные реализации.

Одной из самых популярных библиотек для работы с токенизаторами является **HuggingFace Tokenizers**. Она написана на Rust с привязками для Python и обеспечивает высокую скорость обучения и токенизации (до нескольких миллионов текстов в секунду). Библиотека поддерживает обучение BPE, WordPiece и Unigram, а также предоставляет доступ к множеству предобученных токенизаторов из экосистемы HuggingFace (BERT, GPT-2, RoBERTa, T5 и др.).

В этой лекции мы рассмотрим, как использовать HuggingFace Tokenizers для:
- обучения собственного токенизатора на заданном корпусе с выбором одного из трёх алгоритмов;
- настройки предобработки текста (нормализация, предварительная токенизация, декодирование);
- загрузки и применения предобученных токенизаторов;
- сравнения скорости и качества разных подходов.

Мы также уделим особое внимание **обучению токенизаторов для русского, татарского и таджикского языков**, поскольку эти языки имеют свои особенности алфавита, морфологии и орфографии. Приведём примеры кода на Python, которые можно адаптировать под собственные задачи.

## 2. Установка и базовые компоненты

Библиотека `tokenizers` устанавливается через pip:


In [ ]:
!pip install tokenizers


Основными строительными блоками являются:

- **Tokenizer** — контейнер, который связывает все компоненты (модель, пре-токенизатор, декодер, нормализатор, пост-процессор) и предоставляет методы `train`, `encode`, `decode`.
- **Модель** (`models`) — определяет алгоритм (BPE, WordPiece, Unigram) и хранит выученный словарь.
- **Пре-токенизатор** (`pre_tokenizers`) — разбивает входную строку на предварительные токены (например, по пробелам или на байты) перед обучением и инференсом.
- **Нормализатор** (`normalizers`) — приводит текст к единому виду (нижний регистр, удаление диакритики, нормализация Unicode).
- **Декодер** (`decoders`) — собирает токены обратно в строку.
- **Тренер** (`trainers`) — управляет процессом обучения модели, содержит гиперпараметры (размер словаря, минимальная частота, специальные токены).
- **Пост-процессор** (`post_processors`) — добавляет специальные токены (например, `[CLS]`, `[SEP]`) после токенизации.

## 3. Обучение BPE токенизатора

### 3.1. Краткое напоминание теории

BPE итеративно объединяет самые частые пары соседних токенов. Начальный словарь состоит из всех символов (или байтов). На каждом шаге выбирается пара $(a,b)$ с максимальной частотой $f(a,b)$, создаётся новый токен $ab$, и все вхождения этой пары заменяются на $ab$. Процесс продолжается, пока размер словаря не достигнет заданного значения.

### 3.2. Практика с HuggingFace Tokenizers

Для обучения BPE используем модель `models.BPE` и тренер `trainers.BpeTrainer`.

**Пример:** Обучим BPE на небольшом списке текстов.




In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Инициализируем токенизатор с моделью BPE
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# Устанавливаем пре-токенизатор на пробелы (можно использовать другие)
tokenizer.pre_tokenizer = Whitespace()

# Тренер: задаём размер словаря, специальные токены, минимальную частоту
trainer = BpeTrainer(vocab_size=5000, min_frequency=2, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

# Обучаем на списке файлов или итераторе текстов
texts = [
    "low lower lowest",
    "low low low",
    "lowering the lowest",
    "The quick brown fox jumps over the lazy dog",
    "hello world"
]
tokenizer.train_from_iterator(texts, trainer=trainer)

# Сохраняем токенизатор
tokenizer.save("bpe_tokenizer.json")


**Пояснения:**
- `BpeTrainer` принимает `vocab_size` — целевой размер словаря (включая специальные токены).
- `min_frequency` — минимальное число вхождений пары, чтобы её можно было слить. Позволяет отсечь редкие пары.
- `special_tokens` — список служебных токенов, которые добавляются в словарь до обучения.
- `train_from_iterator` принимает итератор по текстам (или по батчам текстов). Для больших корпусов можно передавать список файлов.

После обучения можно использовать токенизатор:


In [ ]:
output = tokenizer.encode("lowering")
print(output.tokens)   # ['low', 'e', 'ring'] (примерно, зависит от словаря)
print(output.ids)      # числовые идентификаторы


### 3.3. Дополнительные параметры BPE

- `continuing_subword_prefix` — префикс для подслов, которые не являются первыми в слове. Например, `"##"`.
- `end_of_word_suffix` — суффикс для обозначения конца слова (обычно `"</w>"`).
- `byte_fallback` — если True, то неизвестные символы кодируются как байты (аналог Byte-level BPE), что делает токенизатор способным обрабатывать любые символы.

## 4. Обучение WordPiece токенизатора

### 4.1. Краткое напоминание теории

WordPiece также итеративно сливает пары, но критерий выбора пары — не просто частота, а отношение $f(a,b)/(f(a)f(b))$. Это соответствует максимизации прироста логарифмического правдоподобия корпуса в униграммной модели. При инференсе используется жадное сопоставление слева направо.

### 4.2. Практика с HuggingFace Tokenizers


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = WordPieceTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

texts = [
    "low lower lowest",
    "low low low",
    "lowering the lowest",
    "The quick brown fox jumps over the lazy dog",
    "hello world"
]

tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.save("wordpiece_tokenizer.json")


**Особенности:**
- `WordPieceTrainer` использует другую score-функцию, заложенную в реализации.
- Можно задать `continuing_subword_prefix="##"`, чтобы токены, продолжающие слово, помечались.
- Как и в BPE, поддерживается `min_frequency`.

### 4.3. Сравнение результатов

На одном и том же корпусе BPE и WordPiece могут дать разные словари. WordPiece часто выделяет редкие, но сильно связанные пары раньше, чем BPE. Например, в корпусе `"low lower lowest"` WordPiece сначала объединит `(s,t)`, так как score = 1, а BPE начнёт с `(l,o)`, имеющей частоту 3. Это иллюстрирует, как критерий влияет на итоговое разбиение.

## 5. Обучение Unigram токенизатора

### 5.1. Краткое напоминание теории

Unigram LM строит словарь путём удаления из избыточного набора всех подслов, а не слияния. На каждом шаге оцениваются вероятности подслов с помощью EM-алгоритма, затем удаляются наименее полезные подслова (с минимальной потерей правдоподобия). Процесс продолжается до достижения целевого размера словаря.

### 5.2. Практика с HuggingFace Tokenizers




In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(Unigram())
tokenizer.pre_tokenizer = Whitespace()

trainer = UnigramTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    unk_token="[UNK]"
)

texts = [
    "low lower lowest",
    "low low low",
    "lowering the lowest",
    "The quick brown fox jumps over the lazy dog",
    "hello world"
]

tokenizer.train_from_iterator(texts, trainer=trainer)
tokenizer.save("unigram_tokenizer.json")


**Параметры UnigramTrainer:**
- `vocab_size` — целевой размер.
- `min_frequency` — минимальная частота подслова для включения в начальный словарь.
- `max_piece_length` — максимальная длина подслова (по умолчанию 16).
- `n_sub_iterations` — количество итераций удаления (по умолчанию 2).
- `shuffle` — перемешивание корпуса перед обучением.
- `unk_token` — токен для неизвестных символов.

**Важно:** Unigram использует вероятностную сегментацию. При инференсе можно выбрать Viterbi (наиболее вероятное разбиение) или сэмплирование (используется для регуляризации в моделях типа T5). В HuggingFace Tokenizers по умолчанию используется Viterbi, но при желании можно настроить сэмплирование.

## 6. Использование предобученных токенизаторов

Библиотека HuggingFace предоставляет доступ к множеству предобученных токенизаторов через библиотеку `transformers`. Часто проще использовать готовый токенизатор, чем обучать собственный.

**Примеры загрузки предобученных токенизаторов:**




In [ ]:
from transformers import BertTokenizer, GPT2Tokenizer, T5Tokenizer

# BERT (WordPiece)
bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')
tokens = bert_tok.tokenize("The quick brown fox")
print(tokens)  # ['the', 'quick', 'brown', 'fox'] (с префиксами ## для подслов)

# GPT-2 (Byte-level BPE)
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
tokens = gpt2_tok.tokenize("The quick brown fox")
print(tokens)  # ['The', ' quick', ' brown', ' fox'] (пробелы в токенах)

# T5 (Unigram/SentencePiece)
t5_tok = T5Tokenizer.from_pretrained('t5-small')
tokens = t5_tok.tokenize("The quick brown fox")
print(tokens)  # ['▁The', '▁quick', '▁brown', '▁fox'] (нижнее подчёркивание как маркер пробела)


Каждый предобученный токенизатор имеет свой словарь, правила нормализации и пост-обработки. Важно понимать, что выбор токенизатора влияет на вход модели, поэтому нельзя смешивать токенизаторы разных моделей без необходимости.

## 7. Обучение токенизатора для русского, татарского и таджикского языков

Теперь перейдём к практическому вопросу: как обучить токенизатор для трёх языков, имеющих различные особенности. Этот раздел будет полезен, если вы планируете создать многоязычную модель или отдельные модели для каждого языка.

### 7.1. Особенности языков

- **Русский язык** использует кириллицу (33 буквы). Морфология богатая: склонения, спряжения, приставки и суффиксы. Слова могут быть длинными, часты приставки и окончания.
- **Татарский язык** также использует кириллицу (в России) с дополнительными буквами (ә, ө, ү, җ, ң, һ). Агглютинативный строй: цепочки аффиксов, выражающих падеж, число, принадлежность и др.
- **Таджикский язык** официально использует кириллицу (в Таджикистане), но также встречается арабская графика (в Афганистане). Мы будем ориентироваться на кириллический вариант. Таджикский — иранский язык, также агглютинативный, с множеством аффиксов.

Все три языка имеют значительную степень словоизменения, поэтому субсловная токенизация особенно полезна: она позволяет выделять морфемы, что уменьшает размер словаря и улучшает обработку редких слов.

### 7.2. Подготовка корпуса

Для обучения токенизатора необходим достаточно большой текстовый корпус на всех трёх языках. Рекомендуется:
- Использовать смешанный корпус, содержащий тексты на русском, татарском и таджикском. Соотношение может быть равным или соответствовать ожидаемому распределению в приложении.
- Выполнить очистку текста: удалить лишние пробелы, HTML-теги, знаки препинания оставить как есть (они важны).
- Привести текст к нижнему регистру (опционально, но для этих языков обычно полезно, так как уменьшает словарь и упрощает нормализацию).
- Убедиться, что текст в кодировке UTF-8.

### 7.3. Выбор алгоритма и параметров

Для агглютинативных и флективных языков хорошо подходят:
- **BPE** — простой и быстрый, хорошо работает.
- **Unigram** — часто даёт более качественные словари за счёт глобальной оптимизации, но обучение медленнее.
- **WordPiece** — также допустим, но его преимущество перед BPE не так велико.

Для **учебных целей и наглядности** мы выберем **символьный BPE с пробельным пре-токенизатором** (`Whitespace`). Это даст читаемые подслова (например, `при` + `вет`), что облегчает понимание. Если в будущем потребуется обрабатывать произвольные символы (включая арабскую графику), можно перейти на Byte-level BPE, но это выходит за рамки данного примера.

Параметры:
- `vocab_size` = 50 000 (или 30 000 для экономии). Для демонстрационного крошечного корпуса уменьшите до 300.
- `min_frequency` = 2 (или 1, если корпус мал).
- Специальные токены: `[UNK]`, `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`.


### 7.4. Пример кода для обучения на трёх языках




In [ ]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Создаём папку и файлы с примерами
os.makedirs("corpus", exist_ok=True)
with open("corpus/russian_corpus.txt", "w", encoding="utf-8") as f:
    f.write("Привет, как дела?\nЭто пример русского текста для обучения токенизатора.\n")
with open("corpus/tatar_corpus.txt", "w", encoding="utf-8") as f:
    f.write("Сәлам, хәлләрең ничек?\nБу татар телендәге текст мисалы.\n")
with open("corpus/tajik_corpus.txt", "w", encoding="utf-8") as f:
    f.write("Салом, аҳвол чӣ хел?\nИн матни тоҷикӣ барои омӯзиши токенизатор.\n")

# Инициализируем символьный BPE с пробельным пре-токенизатором
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=300,          # для примера; для реального корпуса ставьте 50000
    min_frequency=1,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

files = [
    "corpus/russian_corpus.txt",
    "corpus/tatar_corpus.txt",
    "corpus/tajik_corpus.txt"
]
tokenizer.train(files, trainer=trainer)
tokenizer.save("multilingual_bpe_tokenizer.json")
print("Токенизатор сохранён.")


**Пояснения:**
- `Whitespace()` разбивает текст по пробелам, после чего каждый токен обрабатывается как последовательность символов.
- Кириллические буквы (включая дополнительные татарские и таджикские) являются обычными символами и попадают в словарь.
- `vocab_size` уменьшен до 300 из-за маленького демонстрационного корпуса. При наличии большого корпуса используйте 50 000.

---

### 7.5. Использование SentencePiece как альтернативы

Если вы предпочитаете SentencePiece (например, для унификации с моделями T5 или ALBERT), можно обучить модель так:



In [ ]:
import os
import sentencepiece as spm

# Создаём папку и файлы с примерами
os.makedirs("corpus", exist_ok=True)
with open("corpus/russian.txt", "w", encoding="utf-8") as f:
    f.write("Привет, как дела?\nЭто пример русского текста для обучения токенизатора.\n")
with open("corpus/tatar.txt", "w", encoding="utf-8") as f:
    f.write("Сәлам, хәлләрең ничек?\nБу татар телендәге текст мисалы.\n")
with open("corpus/tajik.txt", "w", encoding="utf-8") as f:
    f.write("Салом, аҳвол чӣ хел?\nИн матни тоҷикӣ барои омӯзиши токенизатор.\n")

spm.SentencePieceTrainer.train(
    input=["corpus/russian.txt", "corpus/tatar.txt", "corpus/tajik.txt"],
    model_prefix="multilingual_spm",
    vocab_size=300,                  # уменьшено для примера; для реального корпуса 50000
    model_type="bpe",
    character_coverage=0.9995,
    byte_fallback=True,
    max_sentence_length=4192,
    input_sentence_size=10000000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

print("Обучение завершено.")


После обучения модель загружается и используется аналогично токенизатору HuggingFace.



In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("multilingual_spm.model")
print(sp.encode("привет", out_type=str))



### 7.6. Обработка таджикского языка с арабской графикой

Если в корпусе встречается таджикский в арабской графике (или вы хотите поддержать её), можно:
- Использовать **Byte-level BPE** (пре-токенизатор `ByteLevel` и модель BPE с `byte_fallback=True`), который автоматически обрабатывает любые символы.
- Либо обучить отдельный токенизатор для арабской графики.
- В нашем примере используется только кириллическая графика, поэтому Byte-level не требуется.



### 7.7. Проверка результатов

После обучения полезно проверить, как токенизатор сегментирует слова на каждом языке. Пример:




In [ ]:
tokenizer = Tokenizer.from_file("multilingual_bpe_tokenizer.json")

for word in ["привет", "сәлам", "салом", "китап", "китоб"]:
    output = tokenizer.encode(word)
    print(f"{word} -> {output.tokens}")


Ожидается, что слова разбиваются на осмысленные подслова, например, `привет` может остаться целым, а `сәлам` может разбиться на `с` + `ә` + `лам`.

## 8. Сравнение скорости и качества

### 8.1. Скорость

Библиотека HuggingFace Tokenizers написана на Rust, что обеспечивает высокую скорость. По тестам разработчиков, она способна токенизировать до 1-2 миллионов текстов в секунду на одном ядре. Для сравнения, реализация на чистом Python обычно на порядок медленнее. При обучении на корпусах объёмом в миллиарды слов использование этой библиотеки практически безальтернативно.

### 8.2. Качество сегментации

Качество субсловной сегментации субъективно, но можно использовать некоторые метрики:
- **Средняя длина токена** (в символах или байтах) — отражает компактность.
- **Доля слов, разбитых на подслова** — показывает, насколько часто модель вынуждена дробить слова.
- **Покрытие словаря** — насколько хорошо словарь покрывает тестовый корпус.

На практике BPE и WordPiece дают похожие результаты, но WordPiece может быть чуть лучше для языков с богатой морфологией за счёт более обоснованного критерия. Unigram, благодаря вероятностной оптимизации, часто даёт более компактные словари при том же размере.

### 8.3. Таблица сравнения

| Критерий               | BPE              | WordPiece        | Unigram           |
|------------------------|------------------|------------------|-------------------|
| Скорость обучения      | Высокая          | Высокая          | Средняя (EM)      |
| Скорость инференса     | Очень высокая    | Очень высокая    | Высокая           |
| Глобальная оптимизация | Нет (жадно)      | Нет (жадно)      | Да (удаление)     |
| Вероятностная сегментация | Нет           | Нет              | Да                |
| Используется в моделях | GPT, RoBERTa     | BERT, DistilBERT | T5, ALBERT, XLNet |

## 9. Практические рекомендации

1. **Всегда добавляйте специальные токены** (`[UNK]`, `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`) с самого начала обучения. Если их не добавить, их придётся добавлять вручную после обучения, что может привести к несоответствию индексов.

2. **Выбирайте размер словаря на основе задачи.** Для английского языка обычно достаточно 30 000–50 000 токенов. Для многоязычных моделей (например, mBERT) используют 100 000–120 000. Для русского, татарского и таджикского можно начать с 50 000, если корпус смешанный.

3. **Настройте `min_frequency`.** Это отсекает редкие пары/подслова, что уменьшает словарь и улучшает обобщение. Обычно значение 2–5.

4. **Используйте подходящий пре-токенизатор.** Для языков с пробелами (русский, татарский, таджикский) подходит `Whitespace`, но для универсальности и работы с редкими символами лучше `ByteLevel` или `Metaspace`.

5. **Нормализуйте текст.** Это включает приведение к нижнему регистру, удаление диакритики, замену редких символов на `[UNK]`. Однако для татарского и таджикского важно сохранить дополнительные буквы (ә, ө, ү и др.), поэтому нельзя просто удалять диакритику. Нормализация Unicode может быть полезна.

6. **Проверьте декодирование.** Убедитесь, что после `encode` и `decode` текст восстанавливается корректно (с точностью до пробелов и нормализации). Это особенно важно для моделей, которые чувствительны к регистру.

7. **Сохраняйте токенизатор в JSON.** Это позволяет легко загружать его позже и делиться с другими.

## 10. Заключение

Библиотека HuggingFace Tokenizers предоставляет мощный и гибкий инструментарий для обучения и использования субсловных токенизаторов. В этой лекции мы рассмотрели, как обучить BPE, WordPiece и Unigram на собственном корпусе, как использовать предобученные токенизаторы из HuggingFace, и дали практические рекомендации. Мы также подробно остановились на обучении токенизаторов для русского, татарского и таджикского языков, учитывая их алфавитные и морфологические особенности. Выбор конкретного алгоритма зависит от задачи, языка и доступных вычислительных ресурсов, но в большинстве случаев все три метода дают приемлемое качество. Понимание их различий позволяет сделать осознанный выбор.

В следующей лекции мы перейдём к сравнительному анализу субсловных методов на реальном корпусе и рассмотрим влияние токенизации на качество downstream-задач.

#  SentencePiece — универсальный инструмент субсловной токенизации

## 1. Введение

В предыдущих лекциях мы подробно рассмотрели алгоритмы BPE, WordPiece и Unigram Language Model, а также их реализацию в библиотеке HuggingFace Tokenizers. Однако на практике широко используется ещё один инструмент — **SentencePiece**, разработанный Таку Кудо и представленный в работе «SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing» (Kudo and Richardson, 2018). SentencePiece задумывался как **универсальный и независимый от языка** токенизатор, который не требует предварительной токенизации на слова. Он напрямую работает с сырым текстом, рассматривая пробелы как обычные символы, и позволяет обучать модели BPE, Unigram и WordPiece (в ограниченном виде).

SentencePiece стал стандартом де-факто для многих современных моделей, особенно многоязычных: T5, ALBERT, XLNet, mBART и др. Его главные преимущества:

- **Языковая независимость**: не нужно разбивать текст на слова, что критично для языков без пробелов (китайский, японский, тайский).
- **Единый формат**: токенизация и детокенизация обратимы благодаря специальному метасимволу пробела `▁` (U+2581).
- **Обработка редких символов**: поддержка `byte_fallback` позволяет кодировать любые символы через их UTF-8 байты, не теряя информацию.
- **Гибкость**: можно выбирать алгоритм (BPE или Unigram) и управлять размером словаря и покрытием символов.

В этой лекции мы детально разберём SentencePiece: его принципы, математические основы (в той мере, в какой они отличаются от уже рассмотренных), параметры обучения, практические примеры для русского, татарского и таджикского языков, а также сравним с HuggingFace Tokenizers.

## 2. Основные идеи SentencePiece

### 2.1. Отказ от предварительной токенизации

Большинство токенизаторов (BERT, GPT-2) ожидают, что входной текст уже разбит на слова (например, по пробелам). Это создаёт проблемы для языков, где пробелы не разделяют слова, или когда мы хотим обрабатывать текст как непрерывный поток символов. SentencePiece решает эту проблему, **включая пробел в алфавит** наравне с буквами. При обучении текст рассматривается как одна длинная последовательность символов (или байтов), и алгоритм может сливать пробел с соседними символами, образуя токены, начинающиеся с пробела. На практике пробел заменяется специальным символом `▁` (U+2581), который визуально отличим и позволяет при декодировании однозначно восстановить пробелы.

### 2.2. Метасимвол пробела `▁`

Вместо обычного пробела SentencePiece использует символ `▁`. Это сделано для наглядности и однозначности: если токен начинается с `▁`, значит перед ним в исходном тексте был пробел. При декодировании все `▁` заменяются на обычные пробелы, а пробелы между токенами не вставляются. Таким образом, достигается полная обратимость: `encode` и `decode` возвращают исходный текст.

Пример: слово `"привет"` после токенизации может быть представлено как `▁привет` (один токен) или `▁при вет` (два токена), где `▁` обозначает границу слова.

### 2.3. Обработка неизвестных символов: byte_fallback

Классические субсловные токенизаторы используют специальный токен `<unk>` для символов, не встретившихся в обучении. SentencePiece может работать в двух режимах:

- **Символьный режим** (`byte_fallback=False`): если символ отсутствует в словаре, он заменяется на `<unk>`.
- **Байтовый режим** (`byte_fallback=True`): неизвестные символы кодируются побайтово с использованием специальных токенов вида `<0xXX>`, соответствующих каждому из 256 байтов. Это гарантирует, что любой символ может быть представлен, даже если он не встречался в обучающем корпусе. Такой подход аналогичен Byte-level BPE, но реализован внутри SentencePiece.

Байтовый режим особенно полезен для многоязычных моделей, где могут встречаться редкие буквы или эмодзи.

### 2.4. Покрытие символов: character_coverage

Параметр `character_coverage` определяет, какую долю символов из обучающего корпуса нужно включить в словарь как отдельные токены (или их части). Например, значение `0.9995` означает, что все символы, встречающиеся с частотой выше 0.05% (т.е. практически все, кроме очень редких), будут добавлены в словарь. Для языков с богатым алфавитом (кириллица, азиатские иероглифы) обычно ставят `0.9995` или `1.0`. Для английского можно `0.999` или выше. Если символ не попал в покрытие, он может быть обработан через `byte_fallback`.

## 3. Математические основы SentencePiece

SentencePiece реализует два основных алгоритма: **BPE** и **Unigram Language Model**. Также в некоторых версиях поддерживается **WordPiece** (через параметр `model_type="word"`), но он менее распространён. Математические основы BPE и Unigram уже подробно описаны в предыдущих лекциях. Здесь мы напомним ключевые формулы и укажем особенности их применения в SentencePiece.

### 3.1. BPE в SentencePiece

BPE итеративно объединяет самую частую пару соседних токенов. В контексте SentencePiece, где текст рассматривается как непрерывная последовательность (включая пробелы как символы), алгоритм работает так же, как классический BPE, но начальный словарь может быть настроен через `character_coverage`. Формула частоты пары $(a,b)$:

$$ f(a,b) = \text{количество вхождений соседних токенов } a \text{ и } b. $$

На каждом шаге выбирается пара $(a^*, b^*)$ с максимальной частотой:

$$ (a^*, b^*) = \arg\max_{a,b} f(a,b). $$

Затем создаётся новый токен $ab$, добавляется в словарь, и все вхождения пары заменяются на $ab$. Процесс продолжается, пока размер словаря не достигнет заданного `vocab_size`.

**Особенность SentencePiece:** поскольку пробелы включены как символы, BPE может сливать пробел с буквами, образуя токены с `▁` в начале. Это позволяет выделять частые слова с пробелами (например, `▁the`).

### 3.2. Unigram LM в SentencePiece

Unigram Language Model строит словарь путём удаления из избыточного набора подслов. Сначала создаётся начальный словарь из всех подстрок, встречающихся в корпусе (до максимальной длины, ограниченной параметром `max_piece_length`), затем с помощью EM-алгоритма оцениваются вероятности подслов, и наименее полезные подслова удаляются.

Пусть словарь $V$ содержит подслова $x$ с вероятностями $p(x)$, $\sum_{x \in V} p(x) = 1$. Вероятность слова (или текста, представленного как последовательность символов) определяется суммой вероятностей всех возможных сегментаций:

$$ P(w) = \sum_{\mathbf{s} \in S(w)} \prod_{i=1}^{|\mathbf{s}|} p(s_i). $$

Логарифмическое правдоподобие корпуса:

$$ \mathcal{L} = \sum_{w \in C} \log P(w). $$

**EM-алгоритм** для оценки $p(x)$:

- **E-шаг:** для каждого слова $w$ вычисляем апостериорные вероятности сегментаций:

  $$ P(\mathbf{s}|w) = \frac{\prod_i p(s_i)}{\sum_{\mathbf{s}'} \prod_j p(s'_j)}. $$

  Ожидаемая частота подслова $x$:

  $$ E(x) = \sum_{w} c(w) \sum_{\mathbf{s}} \left( \sum_i \mathbf{1}_{s_i = x} \right) P(\mathbf{s}|w). $$

- **M-шаг:** обновляем вероятности:

  $$ p(x) = \frac{E(x)}{\sum_{x'} E(x')}. $$

**Удаление подслов:** после EM для каждого подслова вычисляется потеря правдоподобия при его удалении (приближённо $E(x) \log p(x)$), и удаляются наименее полезные (обычно заданная доля $\eta$). Процесс повторяется, пока не достигнут целевой размер словаря.

В SentencePiece `model_type="unigram"` включает этот алгоритм. Параметры `n_sub_iterations` и `shrinking_factor` управляют скоростью удаления.

## 4. Практика: обучение SentencePiece для русского, татарского и таджикского

### 4.1. Подготовка корпуса

Как и для HuggingFace Tokenizers, корпус должен быть текстовым файлом (или несколькими) в UTF-8. Рекомендуется очистить текст от шума, но SentencePiece может работать с сырым текстом.

### 4.2. Пример обучения BPE с SentencePiece




In [ ]:
import os
import sentencepiece as spm

# 1. Создаём папку и файлы с примерами текста
os.makedirs("corpus", exist_ok=True)

with open("corpus/russian.txt", "w", encoding="utf-8") as f:
    f.write("Привет, как дела?\nЭто пример русского текста для обучения токенизатора.\n")
with open("corpus/tatar.txt", "w", encoding="utf-8") as f:
    f.write("Сәлам, хәлләрең ничек?\nБу татар телендәге текст мисалы.\n")
with open("corpus/tajik.txt", "w", encoding="utf-8") as f:
    f.write("Салом, аҳвол чӣ хел?\nИн матни тоҷикӣ барои омӯзиши токенизатор.\n")

# 2. Обучаем SentencePiece BPE
spm.SentencePieceTrainer.train(
    input=["corpus/russian.txt", "corpus/tatar.txt", "corpus/tajik.txt"],
    model_prefix="multilingual_spm_bpe",
    vocab_size=300,                  # уменьшено для демонстрации
    model_type="bpe",
    character_coverage=0.9995,
    byte_fallback=True,
    max_sentence_length=4192,
    input_sentence_size=10000000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

print("Обучение завершено.")

# 3. Загружаем модель и проверяем сегментацию
sp = spm.SentencePieceProcessor()
sp.load("multilingual_spm_bpe.model")

for word in ["привет", "сәлам", "салом", "китап", "китоб"]:
    pieces = sp.encode(word, out_type=str)
    print(f"{word} -> {pieces}")


**Пояснения параметров:**

- `input`: список файлов с текстами.
- `model_prefix`: префикс для выходных файлов (`multilingual_spm_bpe.model` и `.vocab`).
- `vocab_size`: целевой размер словаря (включая специальные токены).
- `model_type`: `"bpe"`, `"unigram"` (или `"word"` для WordPiece, но в новых версиях может быть `"word"`).
- `character_coverage`: доля символов из корпуса, которые будут включены в словарь как отдельные токены. Для кириллицы с дополнительными буквами рекомендуется 0.9995 или 1.0.
- `byte_fallback`: если `True`, неизвестные символы кодируются байтами (токены вида `<0xXX>`), что делает токенизатор универсальным.
- `max_sentence_length`: максимальная длина входной строки (в символах) для обработки. Большие строки обрезаются.
- `input_sentence_size`: количество предложений, считываемых для обучения (если корпус больше, берётся случайная подвыборка).
- `shuffle_input_sentence`: перемешивать ли предложения перед обучением.
- `pad_id`, `unk_id`, `bos_id`, `eos_id`: индексы специальных токенов. Обычно `pad=0`, `unk=1`, `bos=2`, `eos=3`.

### 4.3. Пример обучения Unigram с SentencePiece




In [ ]:
import os
import sentencepiece as spm

# 1. Создаём папку и файлы с примерами текста
os.makedirs("corpus", exist_ok=True)

with open("corpus/russian.txt", "w", encoding="utf-8") as f:
    f.write("Привет, как дела?\nЭто пример русского текста для обучения токенизатора.\n")
with open("corpus/tatar.txt", "w", encoding="utf-8") as f:
    f.write("Сәлам, хәлләрең ничек?\nБу татар телендәге текст мисалы.\n")
with open("corpus/tajik.txt", "w", encoding="utf-8") as f:
    f.write("Салом, аҳвол чӣ хел?\nИн матни тоҷикӣ барои омӯзиши токенизатор.\n")

# 2. Обучаем SentencePiece Unigram
spm.SentencePieceTrainer.train(
    input=["corpus/russian.txt", "corpus/tatar.txt", "corpus/tajik.txt"],
    model_prefix="multilingual_spm_unigram",
    vocab_size=300,
    model_type="unigram",
    character_coverage=0.9995,
    byte_fallback=True,
    max_sentence_length=4192,
    input_sentence_size=10000000,
    shuffle_input_sentence=True,
    unk_id=1, bos_id=2, eos_id=3, pad_id=0
)

print("Обучение завершено.")

# 3. Загружаем модель и проверяем сегментацию
sp = spm.SentencePieceProcessor()
sp.load("multilingual_spm_unigram.model")

for word in ["привет", "сәлам", "салом", "китап", "китоб"]:
    pieces = sp.encode(word, out_type=str)
    print(f"{word} -> {pieces}")


Разница только в `model_type="unigram"`. Unigram обычно требует больше вычислительных ресурсов, но может дать более качественный словарь.

### 4.4. Загрузка и использование модели




In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("multilingual_spm_bpe.model")

# Кодирование
text = "Привет, как дела?"
pieces = sp.encode(text, out_type=str)
print(pieces)  # ['▁Привет', ',', '▁как', '▁дела', '?']

# Декодирование
decoded = sp.decode(pieces)
print(decoded)  # "Привет, как дела?"

# Получение идентификаторов
ids = sp.encode(text, out_type=int)
print(ids)


Обратите внимание на `▁` перед словами. При декодировании `decode` автоматически преобразует `▁` в пробелы и убирает лишние пробелы.

### 4.5. Проверка для наших языков




In [ ]:
for word in ["привет", "сәлам", "салом", "китап", "китоб"]:
    pieces = sp.encode(word, out_type=str)
    print(f"{word} -> {pieces}")



Это читаемые подслова, что делает SentencePiece удобным для анализа.

## 5. Сравнение SentencePiece и HuggingFace Tokenizers

| Характеристика               | SentencePiece                    | HuggingFace Tokenizers           |
|------------------------------|----------------------------------|----------------------------------|
| Языковая зависимость         | Не зависит от языка (работает с сырым текстом) | Требует пре-токенизатора (Whitespace, ByteLevel и т.д.) |
| Пробелы                      | Метасимвол `▁`                   | Обычно пробелы удаляются или включаются в токены (ByteLevel) |
| byte_fallback                | Встроен, удобен                  | Достигается через ByteLevel пре-токенизатор |
| Алгоритмы                    | BPE, Unigram, WordPiece (ограничено) | BPE, WordPiece, Unigram         |
| Скорость                     | Высокая (C++)                    | Очень высокая (Rust)            |
| Использование                | T5, ALBERT, XLNet                | BERT, GPT-2, RoBERTa и др.      |
| Формат сохранения            | `.model` и `.vocab`              | `.json` (один файл)             |
| Гибкость предобработки       | Встроенная нормализация          | Раздельные компоненты (Normalizer, PreTokenizer и др.) |

**Когда выбирать SentencePiece:**
- Если модель обучается с нуля и нужна универсальность для многих языков, включая языки без пробелов.
- Если вы используете модели, которые уже обучены с SentencePiece (T5, ALBERT), и хотите сохранить совместимость.
- Если вы хотите простой интерфейс без необходимости настраивать несколько компонентов.

**Когда выбирать HuggingFace Tokenizers:**
- Если вы используете готовые модели из HuggingFace и вам нужно точно воспроизвести их токенизацию.
- Если вам нужна максимальная скорость и гибкость в настройке пре-токенизатора, нормализатора, пост-процессора.
- Если вы обучаете новую модель и хотите использовать Rust-реализацию для быстрого прототипирования.

## 6. Заключение

SentencePiece — мощный и удобный инструмент для субсловной токенизации, который устраняет необходимость в предварительном разбиении на слова и обеспечивает универсальность для любого языка. Благодаря метасимволу `▁`, байтовому фолбэку и гибким параметрам (`character_coverage`, `vocab_size`, `model_type`) он широко используется в современных моделях. В этой лекции мы рассмотрели его основные принципы, математические аспекты (BPE и Unigram), практические примеры обучения для русского, татарского и таджикского языков и сравнили с HuggingFace Tokenizers. Выбор между ними зависит от конкретной задачи, но знание обоих инструментов позволяет принимать взвешенные решения.


# Проблемы и ограничения субсловной токенизации. Современные тенденции: токенизаторы без словаря

## 1. Введение

В предыдущих лекциях мы подробно рассмотрели основные алгоритмы субсловной токенизации: BPE, Byte-level BPE, WordPiece и Unigram Language Model. Они стали стандартом в современных нейронных моделях обработки естественного языка, позволив преодолеть ограничения пословной токенизации и обеспечить компактные словари с возможностью обрабатывать редкие и неизвестные слова. Однако, несмотря на широкое распространение и практическую эффективность, субсловная токенизация не лишена недостатков. Эти ограничения становятся особенно заметными при работе с определёнными типами текстов (числа, эмодзи, смешение языков) и стимулируют развитие новых подходов, включая модели, вообще не использующие фиксированный словарь.

В этой лекции мы систематизируем основные проблемы и ограничения субсловной токенизации, а затем обратимся к современным тенденциям — tokenization-free моделям, которые пытаются обрабатывать текст напрямую на уровне символов или байтов, избегая недостатков словарных методов.

## 2. Проблемы и ограничения субсловной токенизации

### 2.1. Числа и даты

Одна из наиболее известных проблем BPE и подобных методов — неадекватное разбиение чисел. Поскольку числа могут быть произвольной длины и встречаются в тексте с разной частотой, алгоритм часто делит их на бессмысленные фрагменты. Например, число `1234567890` после обучения BPE на типичном корпусе может быть разбито как `123 456 789 0` или даже `1 23 45 67 89 0`. Это происходит потому, что частые пары цифр (например, `12`, `45`) сливаются в токены, но полное число почти никогда не попадает в словарь целиком из-за бесконечного разнообразия числовых значений.

Такое разбиение плохо тем, что модель теряет представление о числовой величине. В задачах, где числа важны (математические вычисления, работа с датами, денежными суммами), это может серьёзно ухудшить качество. Например, для модели, которая должна ответить на вопрос «Сколько будет 123456 + 789?», разбиение числа на части `123 456` и `789` может привести к путанице, поскольку модель будет обрабатывать эти фрагменты как независимые токены, не видя целого числа.

**Иллюстрация.** Предположим, обученный BPE имеет в словаре токены `123`, `456`, `789`. Тогда число `123456789` закодируется как `[123][456][789]`. При этом модель может ошибочно интерпретировать `123` как отдельное число, а не как часть большего.

Некоторые современные подходы пытаются решить эту проблему путём специальной обработки чисел (например, замены их на псевдотокены вроде `<NUM>`), но это требует дополнительной предобработки и не всегда применимо.

### 2.2. Регистр и пунктуация

Классические субсловные токенизаторы чувствительны к регистру. Если в словаре есть слово `Apple` и слово `apple`, они будут разными токенами. Это удваивает количество токенов для слов, которые встречаются в обоих регистрах. Многие модели (например, BERT uncased) решают эту проблему путём приведения всех букв к нижнему регистру перед токенизацией, но это приводит к потере информации, что может быть критично для задач, где регистр значим (например, распознавание именованных сущностей: `Apple` как компания против `apple` как фрукт).

Пунктуация также представляет сложность. Знаки препинания (`.`, `,`, `!`, `?`) часто рассматриваются как отдельные токены, но они могут быть частью сокращений (`e.g.`, `Mr.`), чисел (`3.14`), эмодзи (`:)`) и т.д. BPE может некорректно отделять точку от слова, что приводит к появлению лишних токенов и усложняет декодирование.

### 2.3. Эмодзи и специальные символы

Эмодзи и другие специальные символы (пиктограммы, математические знаки, символы валют) часто отсутствуют в обучающем корпусе или встречаются очень редко. В символьных BPE/WordPiece они обычно заменяются на `<unk>`, что означает полную потерю информации. Byte-level BPE решает эту проблему, представляя каждый символ через его UTF-8 байты, что гарантирует кодирование любого символа, но при этом байтовые токены могут быть неинтерпретируемы и удлиняют последовательность.

Пример: эмодзи 😀 в UTF-8 кодируется четырьмя байтами `F0 9F 98 80`. В Byte-level BPE он будет представлен, скорее всего, как один или несколько токенов, соответствующих этим байтам. Модель, обученная на текстах с эмодзи, может выучить их значение, но если эмодзи встречается редко, его байтовое представление будет разбито на части, что также не идеально.

### 2.4. Омонимия и неоднозначность сегментации

Субсловные методы порождают неоднозначность: одно и то же слово может быть разбито на подслова несколькими способами, и выбор конкретного разбиения может влиять на смысл. Например, английское слово `unhappiness` может быть сегментировано как `un happiness` или `unhappi ness` (в зависимости от выученного словаря). BPE и WordPiece используют жадное разбиение, которое не всегда совпадает с морфемными границами. Это может привести к тому, что модель будет видеть разные морфемы в разных контекстах, что усложняет обучение.

Unigram LM, благодаря вероятностной модели, может вычислять несколько возможных сегментаций и даже сэмплировать их, что частично смягчает проблему, но всё равно не гарантирует морфологическую правильность.

### 2.5. Морфологическая несогласованность для агглютинативных языков

Для языков с богатой морфологией (русский, татарский, турецкий, финский) субсловная токенизация часто выделяет подслова, не соответствующие настоящим морфемам. Например, русское слово `наибыстрейшему` может быть разбито как `наи быстрей шему`, что не совпадает с лингвистическим разбором. Это может ухудшить способность модели к обобщению, особенно при работе с редкими формами слов.

### 2.6. Проблема редких слов

Хотя субсловная токенизация решает проблему неизвестных слов, она не устраняет её полностью: если слово очень редкое, оно может быть разбито на очень мелкие части (вплоть до отдельных символов). Это делает последовательность длинной и может затруднить обучение. Например, термин `гиппопотомонстросесквипедалиофобия` будет разбит на множество токенов, что увеличит вычислительные затраты и может ухудшить представление слова.

### 2.7. Влияние на вычислительные ресурсы

Субсловная токенизация увеличивает длину последовательностей по сравнению с пословной. Для каждого слова в среднем получается 1.5–2 токена, что ведёт к росту потребления памяти и времени вычислений в моделях на основе Transformer (сложность $O(n^2)$ по длине последовательности). Это особенно критично для длинных документов.

### 2.8. Проблемы многоязычности

При обучении одного токенизатора на нескольких языках с разными алфавитами (кириллица, латиница, арабская графика) словарь должен включать символы всех алфавитов. Это увеличивает размер словаря и может привести к неоптимальному распределению токенов между языками. Например, если корпус на 90% состоит из английского и на 10% из русского, то русские слова будут разбиваться на более мелкие части, так как частоты русских букв ниже. Это ставит языки в неравное положение.

## 3. Современные тенденции: токенизаторы без словаря

Перечисленные проблемы стимулировали исследования, направленные на создание моделей, которые не полагаются на фиксированный словарь подслов. Вместо этого они обрабатывают текст на уровне отдельных символов или байтов, полагаясь на способность нейронной сети самостоятельно выучивать необходимые комбинации. Такие подходы получили название **tokenization-free** или **subword-agnostic**.

### 3.1. Модели на уровне байтов: ByT5, CANINE

Модель **ByT5** (Xue et al., 2021) — это вариант T5, который работает непосредственно с UTF-8 байтами. Вместо токенизатора текст разбивается на последовательность байтов, каждый байт отображается в эмбеддинг. ByT5 показал сопоставимое качество с моделями на основе SentencePiece в задачах классификации, QA и генерации, но при этом имеет фиксированный размер словаря (256) и не требует обучения токенизатора. Недостаток — последовательности становятся очень длинными (в несколько раз длиннее, чем при субсловной токенизации), что увеличивает вычислительные затраты.

**CANINE** (Clark et al., 2022) — модель, также работающая на уровне символов, но использующая свёрточные слои для локального кодирования символов перед подачей в Transformer. Это позволяет сократить длину последовательности, агрегируя символы в блочные представления, и достичь производительности, близкой к моделям с токенизатором. CANINE применялся к задачам на разных языках и показал устойчивость.

### 3.2. Модели на уровне символов: Charformer, TransformerXL на символах

Ряд работ исследует возможность обучения Transformer прямо на последовательностях символов. Например, **Charformer** (Tay et al., 2021) вводит обучаемые параметры, которые автоматически группируют символы в подобие подслов на основе градиентов. Модель может динамически выбирать гранулярность: от отдельных символов до целых слов. Это позволяет обойтись без предварительно обученного словаря.

Другой подход — использовать очень глубокие модели с малым числом параметров, которые могут обрабатывать длинные последовательности символов (например, Transformer-XL с рекуррентной памятью). Такие модели уже способны выучивать морфологию на символьном уровне, но требуют значительно больше вычислительных ресурсов.

### 3.3. Гибридные подходы: Mixture-of-Tokens, динамическая токенизация

Некоторые исследователи предлагают гибридные схемы, где токенизация выполняется на лету самой моделью. Например, **Mixture-of-Tokens** (Liu et al., 2023) использует несколько уровней токенизации (байты, символы, подслова) и позволяет модели обучать распределение по ним. Это сочетает гибкость без словаря с эффективностью субсловных методов.

Ещё одно направление — **динамическая токенизация**, когда модель получает не только токены, но и информацию о границах символов, и может адаптивно менять сегментацию в зависимости от контекста.

### 3.4. Перспективы и вызовы

Переход к безсловарным моделям решает многие проблемы субсловной токенизации:
- Нет неизвестных токенов — любой символ может быть представлен.
- Нет необходимости обучать и хранить словарь.
- Универсальность для всех языков.
- Более простое декодирование (символы/байты непосредственно переводятся в текст).

Однако остаются существенные вызовы:
- **Длина последовательностей**: байтовые и символьные модели обрабатывают в 4–10 раз больше токенов, что резко повышает требования к памяти и вычислениям.
- **Качество**: пока субсловные модели часто показывают лучшее качество на ряде задач, особенно генеративных.
- **Обучение**: требуется больше данных и более тщательная настройка для символьных моделей, чтобы они выучили морфологию.

Тем не менее, тенденция движется к уменьшению зависимости от фиксированных словарей. Возможно, будущее за адаптивными методами, которые будут сочетать эффективность подслов с гибкостью символов.

## 4. Заключение

Субсловная токенизация, несмотря на свои ограничения, на сегодняшний день остаётся наиболее практичным и широко используемым подходом. Она обеспечивает хороший баланс между размером словаря, длиной последовательностей и способностью обрабатывать редкие слова. Однако проблемы с числами, эмодзи, морфологической несогласованностью и многоязычностью побуждают исследователей искать альтернативы.

Современные tokenization-free модели, работающие на байтах или символах, предлагают элегантное решение многих из этих проблем, но сталкиваются с вычислительными трудностями. Вероятно, в ближайшие годы мы увидим дальнейшее развитие гибридных подходов, которые объединят преимущества обоих направлений.

В этой лекции мы обсудили ключевые недостатки субсловной токенизации и познакомились с основными тенденциями в области безсловарных моделей. Это завершает наш цикл по токенизации. Понимание этих вопросов необходимо для осознанного выбора архитектуры и предобработки текста в реальных проектах.